In [22]:
from mlflow.tracking import MlflowClient

MLFOW_TRACKING_URI = "sqlite:////home/ubuntu/Mlops-course/03-experiment-tracking/mlflow.db"

client = MlflowClient(tracking_uri=MLFOW_TRACKING_URI)

In [23]:
client.search_experiments()

[<Experiment: artifact_location='/home/ubuntu/Mlops-course/02-ml-model/mlruns/2', creation_time=1784428392824, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1784428392824, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/home/ubuntu/Mlops-course/02-ml-model/mlruns/1', creation_time=1783744047989, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1783744047989, lifecycle_stage='active', name='nyc_taxi_experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1783737023715, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1783737023715, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [24]:
client.create_experiment(name="my-cool-experiment")

MlflowException: Experiment(name=my-cool-experiment) already exists. Error: (sqlite3.IntegrityError) UNIQUE constraint failed: experiments.workspace, experiments.name
[SQL: INSERT INTO experiments (name, workspace, artifact_location, lifecycle_stage, creation_time, last_update_time) VALUES (?, ?, ?, ?, ?, ?)]
[parameters: ('my-cool-experiment', 'default', None, 'active', 1784688094393, 1784688094393)]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

In [25]:
from mlflow.entities import ViewType

runs_con_modelo = client.search_runs(
    experiment_ids='1',
    filter_string="",
    order_by=['start_time DESC'],
    max_results=10
)

In [26]:
import mlflow
for run in runs_con_modelo:
    artifacts = client.list_artifacts(run.info.run_id)
    paths = [a.path for a in artifacts]
    if 'models_mlflow' in paths:
        print(f"✅ run id: {run.info.run_id}, rmse: {run.data.metrics.get('rmse')}")

client.list_artifacts

<bound method MlflowClient.list_artifacts of <mlflow.tracking.client.MlflowClient object at 0x776223293950>>

In [27]:
import mlflow

mlflow.set_tracking_uri(MLFOW_TRACKING_URI)

In [29]:
run_id = 'f16636c84de44fd8bee32cac0bc095c2'
model_uri = f'runs:/{run_id}/models_mlflow' 
mlflow.register_model(model_uri=model_uri,name='nyc_taxi_experiment')

Registered model 'nyc_taxi_experiment' already exists. Creating a new version of this model...
2026/07/22 02:42:52 WARNING mlflow.tracking._model_registry.fluent: Run with id f16636c84de44fd8bee32cac0bc095c2 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-34a9cde4dee14ef4b925d0713f9f5250 instead
Created version '1' of model 'nyc_taxi_experiment'.


<ModelVersion: aliases=[], creation_timestamp=1784688172251, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1784688172251, metrics=None, model_id=None, name='nyc_taxi_experiment', params=None, run_id='f16636c84de44fd8bee32cac0bc095c2', run_link=None, source='models:/m-34a9cde4dee14ef4b925d0713f9f5250', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>

In [30]:
model_uri

'runs:/f16636c84de44fd8bee32cac0bc095c2/models_mlflow'

In [34]:
client.search_registered_models()
model_name = "nyc-taxi-regressor"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 2, stage: None


/tmp/ipykernel_159036/92185721.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=2,
    stage="None",
    archive_existing_versions=False
)